<a href="https://colab.research.google.com/github/mohamedmahdibenoun/capstone-DA-project/blob/main/optimisationCPU_TPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Cell 1 : Install (ONLY if you are on Google Colab with TPU)
# Runtime -> Change runtime type -> TPU
# ============================================================

# Uncomment if needed in Colab:
# !pip install -q cloud-tpu-client
# !pip install -q torch torchvision
# !pip install -q torch_xla[tpu]~=2.2 -f \
#   https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.2.html

import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


In [2]:
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    TPU_AVAILABLE = True
    print("TPU detected ✅", xm.get_xla_supported_devices())
except ImportError:
    TPU_AVAILABLE = False
    print("TPU not available, TPU part will be skipped.")


/usr/local/lib/python3.12/dist-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or


TPU detected ✅ ['xla:0']


In [3]:
BATCH_SIZE_DEFAULT = 128

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True,
                               download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False,
                               download=True, transform=transform)


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 345kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.20MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.8MB/s]


In [4]:
class SimpleCNN(nn.Module):
    # modèle "standard"
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


class SmallCNN(nn.Module):
    # modèle "réduit" pour la méthode CPU : réduction de complexité
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1   = nn.Linear(32 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [5]:
class AdaGradCustom(torch.optim.Optimizer):
    def __init__(self, params, lr=0.01, eps=1e-8):
        defaults = dict(lr=lr, eps=eps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            eps = group['eps']
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]

                if "G" not in state:
                    state["G"] = torch.zeros_like(p.data)

                G = state["G"]
                G.addcmul_(grad, grad, value=1.0)     # G_t+1 = G_t + grad^2
                denom = G.sqrt().add_(eps)
                p.data.addcdiv_(grad, denom, value=-lr)
        return loss


In [6]:
def train_one_epoch(model, device, dataloader, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for data, target in dataloader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += data.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, device, dataloader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for data, target in dataloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = F.cross_entropy(output, target)

        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += data.size(0)

    return total_loss / total, correct / total


In [7]:
def train_cpu(config_name,
              model_class=SimpleCNN,
              batch_size=BATCH_SIZE_DEFAULT,
              num_workers=0,
              pin_memory=False,
              max_threads=None,
              epochs=3):

    print(f"\n===== CPU config: {config_name} =====")

    if max_threads is not None:
        torch.set_num_threads(max_threads)

    device = torch.device("cpu")

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    model = model_class().to(device)
    optimizer = AdaGradCustom(model.parameters(), lr=0.05)

    history = {"train_loss": [], "train_acc": [],
               "val_loss": [], "val_acc": []}

    start = time.time()
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, device, train_loader, optimizer)
        val_loss, val_acc = evaluate(model, device, test_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"[CPU-{config_name}] Epoch {epoch}: "
              f"train_loss={train_loss:.4f}, acc={train_acc:.4f}, "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    total_time = time.time() - start
    print(f"Total CPU time ({config_name}) = {total_time:.2f} s")
    return history, total_time


In [12]:
def convert_model_to_bfloat16(model):
    # convert all parameters AND buffers (weights, biases, running stats...)
    for name, module in model.named_modules():
        for param_name, param in module.named_parameters(recurse=False):
            module._parameters[param_name] = param.to(torch.bfloat16)

        for buffer_name, buffer in module.named_buffers(recurse=False):
            module._buffers[buffer_name] = buffer.to(torch.bfloat16)

    return model


In [15]:
def train_tpu(config_name,
              model_class=SimpleCNN,
              batch_size=BATCH_SIZE_DEFAULT,
              use_bfloat16=False,
              epochs=3):

    if not TPU_AVAILABLE:
        raise RuntimeError("TPU not available in this environment.")

    print(f"\n===== TPU config: {config_name} =====")

    device = xm.xla_device()

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2
    )

    train_device_loader = pl.MpDeviceLoader(train_loader, device)
    test_device_loader  = pl.MpDeviceLoader(test_loader, device)

    # --------- MODEL ---------
    model = model_class().to(device)

    if use_bfloat16:
        # conversion simple et propre : tous les paramètres restent des leaf tensors
        model = model.to(dtype=torch.bfloat16)

    optimizer = AdaGradCustom(model.parameters(), lr=0.05)

    history = {"train_loss": [], "train_acc": [],
               "val_loss": [], "val_acc": []}

    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for data, target in train_device_loader:
            # --------- INPUTS ---------
            data = data.to(device)
            target = target.to(device)

            if use_bfloat16:
                # IMPORTANT : on convertit aussi les entrées en bfloat16
                data = data.to(dtype=torch.bfloat16)

            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            xm.optimizer_step(optimizer, barrier=True)

            total_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += data.size(0)

        train_loss = total_loss / total
        train_acc  = correct / total

        val_loss, val_acc = evaluate(model, device, test_device_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"[TPU-{config_name}] Epoch {epoch}: "
              f"train_loss={train_loss:.4f}, acc={train_acc:.4f}, "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    total_time = time.time() - start
    print(f"Total TPU time ({config_name}) = {total_time:.2f} s")
    return history, total_time


In [21]:
EPOCHS = 3  # tu peux mettre 5 comme dans ton rapport
results = {}

# ----------------- CPU METHODS -----------------
# a) Vectorisation / SIMD -> simplement PyTorch + BLAS, config de base
hist, t = train_cpu("BASE_VECTORISATION",
                    model_class=SimpleCNN,
                    batch_size=BATCH_SIZE_DEFAULT,
                    num_workers=0,   # pas de workers supplémentaires
                    pin_memory=False,
                    max_threads=None,
                    epochs=EPOCHS)
results[("CPU", "Vectorisation")] = (hist, t)

# b) Parallélisation multi-threads
hist, t = train_cpu("MULTI_THREADS",
                    model_class=SimpleCNN,
                    batch_size=BATCH_SIZE_DEFAULT,
                    num_workers=4,   # threads de chargement de données
                    pin_memory=True,
                    max_threads=8,   # plus de threads CPU pour BLAS
                    epochs=EPOCHS)
results[("CPU", "MultiThreads")] = (hist, t)

# c) Optimisation mémoire (cache-aware) -> gros batch + pin_memory
hist, t = train_cpu("CACHE_AWARE",
                    model_class=SimpleCNN,
                    batch_size=256,      # batch plus grand
                    num_workers=2,
                    pin_memory=True,
                    max_threads=8,
                    epochs=EPOCHS)
results[("CPU", "CacheAware")] = (hist, t)

# d) Réduction de complexité algorithmique -> modèle plus petit
hist, t = train_cpu("ALGO_COMPLEXITY",
                    model_class=SmallCNN,
                    batch_size=BATCH_SIZE_DEFAULT,
                    num_workers=2,
                    pin_memory=True,
                    max_threads=8,
                    epochs=EPOCHS)
results[("CPU", "AlgoComplexity")] = (hist, t)


# ----------------- TPU METHODS -----------------
if TPU_AVAILABLE:
    # a) Exploitation du parallélisme massif (MXU) -> config de base
    hist, t = train_tpu("PARALLEL_MXU",
                        model_class=SimpleCNN,
                        batch_size=BATCH_SIZE_DEFAULT,
                        use_bfloat16=False,
                        epochs=EPOCHS)
    results[("TPU", "MXU")] = (hist, t)



    # c) Fusion d'opérations -> même boucle mais via XLA (déjà fait),
    #    on garde une étiquette séparée pour le rapport
    hist, t = train_tpu("FUSION_XLA",
                        model_class=SimpleCNN,
                        batch_size=BATCH_SIZE_DEFAULT,
                        use_bfloat16=False,
                        epochs=EPOCHS)
    results[("TPU", "Fusion")] = (hist, t)

    # d) Pipelining / batch processing -> gros batch
    hist, t = train_tpu("BATCHING",
                        model_class=SimpleCNN,
                        batch_size=512,   # batch beaucoup plus grand
                        use_bfloat16=False,
                        epochs=EPOCHS)
    results[("TPU", "Batching")] = (hist, t)
else:
    print("TPU not available, TPU methods were skipped.")



===== CPU config: BASE_VECTORISATION =====
[CPU-BASE_VECTORISATION] Epoch 1: train_loss=0.8038, acc=0.9199, val_loss=0.0854, val_acc=0.9737
[CPU-BASE_VECTORISATION] Epoch 2: train_loss=0.0749, acc=0.9775, val_loss=0.0583, val_acc=0.9816
[CPU-BASE_VECTORISATION] Epoch 3: train_loss=0.0572, acc=0.9829, val_loss=0.0531, val_acc=0.9841
Total CPU time (BASE_VECTORISATION) = 64.86 s

===== CPU config: MULTI_THREADS =====
[CPU-MULTI_THREADS] Epoch 1: train_loss=0.9818, acc=0.9217, val_loss=0.0857, val_acc=0.9725
[CPU-MULTI_THREADS] Epoch 2: train_loss=0.0695, acc=0.9790, val_loss=0.0600, val_acc=0.9797
[CPU-MULTI_THREADS] Epoch 3: train_loss=0.0525, acc=0.9839, val_loss=0.0464, val_acc=0.9842
Total CPU time (MULTI_THREADS) = 55.89 s

===== CPU config: CACHE_AWARE =====
[CPU-CACHE_AWARE] Epoch 1: train_loss=0.9475, acc=0.8911, val_loss=0.2445, val_acc=0.9265
[CPU-CACHE_AWARE] Epoch 2: train_loss=0.0887, acc=0.9718, val_loss=0.0759, val_acc=0.9747
[CPU-CACHE_AWARE] Epoch 3: train_loss=0.0695, 

/tmp/ipython-input-2917629489.py:12: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


[TPU-PARALLEL_MXU] Epoch 1: train_loss=0.9806, acc=0.8367, val_loss=0.2077, val_acc=0.9316
[TPU-PARALLEL_MXU] Epoch 2: train_loss=0.1927, acc=0.9386, val_loss=0.1638, val_acc=0.9476
[TPU-PARALLEL_MXU] Epoch 3: train_loss=0.1602, acc=0.9497, val_loss=0.1442, val_acc=0.9543
Total TPU time (PARALLEL_MXU) = 17.41 s

===== TPU config: FUSION_XLA =====
[TPU-FUSION_XLA] Epoch 1: train_loss=2.0586, acc=0.9503, val_loss=0.0538, val_acc=0.9814
[TPU-FUSION_XLA] Epoch 2: train_loss=0.0483, acc=0.9851, val_loss=0.0462, val_acc=0.9850
[TPU-FUSION_XLA] Epoch 3: train_loss=0.0367, acc=0.9889, val_loss=0.0385, val_acc=0.9865
Total TPU time (FUSION_XLA) = 17.18 s

===== TPU config: BATCHING =====
[TPU-BATCHING] Epoch 1: train_loss=2.8035, acc=0.8649, val_loss=0.1006, val_acc=0.9669
[TPU-BATCHING] Epoch 2: train_loss=0.0734, acc=0.9777, val_loss=0.0758, val_acc=0.9742
[TPU-BATCHING] Epoch 3: train_loss=0.0554, acc=0.9827, val_loss=0.0547, val_acc=0.9824
Total TPU time (BATCHING) = 16.10 s


In [22]:
def summarize(results):
    print("\n======= Résumé final (temps + accuracy) =======\n")
    print("{:8s} {:18s} {:>10s} {:>10s} {:>10s}".format(
        "Device", "Méthode", "ValAcc", "ValLoss", "Temps(s)"))
    print("-" * 60)
    for (device, method), (hist, t_total) in results.items():
        val_acc  = hist["val_acc"][-1]
        val_loss = hist["val_loss"][-1]
        print("{:8s} {:18s} {:10.4f} {:10.4f} {:10.2f}".format(
            device, method, val_acc, val_loss, t_total))

summarize(results)



======= Résumé final (temps + accuracy) =======

Device   Méthode                ValAcc    ValLoss   Temps(s)
------------------------------------------------------------
CPU      Vectorisation          0.9841     0.0531      64.86
CPU      MultiThreads           0.9842     0.0464      55.89
CPU      CacheAware             0.9791     0.0638      51.28
CPU      AlgoComplexity         0.9846     0.0466      33.55
TPU      MXU                    0.9543     0.1442      17.41
TPU      Fusion                 0.9865     0.0385      17.18
TPU      Batching               0.9824     0.0547      16.10
